# Diagnostic Analytics — Regional Trends, Age, and Gender Patterns

**Goal:** Answer specific business questions to understand regional variations in BMI and smoking, and diagnose how age and gender truly affect medical charges when controlling for smoking status.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('insurance_clean.csv')

In [ ]:
# Create a numeric smoker column (1 for yes, 0 for no) to easily calculate percentages
df['smoker_rate'] = np.where(df['smoker'] == 'yes', 1, 0)

print("Data loaded successfully. Shape:", df.shape)

## 1. Smoker Rate vs. Average Charges by Region 

**Question:** What is the smoker rate by region, and does the region with the highest smoker rate also have the highest average charges?

In [ ]:
bq5_analysis = df.groupby('region')[['smoker_rate', 'charges']].mean().round(4)
bq5_analysis['smoker_rate'] = bq5_analysis['smoker_rate'] * 100
bq5_analysis = bq5_analysis.sort_values(by='smoker_rate', ascending=False)
print(bq5_analysis)

**Finding:** Yes, the region with the highest smoker rate (Southeast at ~25%) is also the region with the highest average charges (~$14,735). This confirms that regional cost differences are strongly driven by how many people smoke in that area.

## 2. Average BMI and Charges by Region

**Question:** What is the average BMI by region, and is there a region that combines above-average BMI with above-average charges?

In [ ]:
overall_bmi = df['bmi'].mean()
overall_charges = df['charges'].mean()

print(f"Overall Average BMI: {overall_bmi:.2f}")
print(f"Overall Average Charges: ${overall_charges:.2f}\n")

In [ ]:
bq6_analysis = df.groupby('region')[['bmi', 'charges']].mean().round(2)
print("=== Regional Averages ===")
print(bq6_analysis)
high_risk = bq6_analysis[(bq6_analysis['bmi'] > overall_bmi) & (bq6_analysis['charges'] > overall_charges)]
print("\n=== Regions with Above-Average BMI AND Charges ===")
print(high_risk)

**Finding:** The Southeast is the only region that combines both above-average BMI (33.36) and above-average charges ($14,735.41). This highlights the Southeast as the highest-risk and highest-cost region in the dataset.

## 3. Age vs. Charges: Steady Increase or Sharp Jump? 

**Question:** Do charges increase at a steady rate as age increases, or do they jump sharply?

In [ ]:
sns.scatterplot(data=df, x='age', y='charges', hue='smoker', palette={'yes': '#e07a5f', 'no': '#3d9970'}, alpha=0.7)
plt.title('Medical Charges vs. Age (Colored by Smoker Status)', fontsize=13, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Charges ($)')
plt.tight_layout()
plt.show()

**Finding:** Charges increase at a steady, constant rate as age increases for non-smokers (the lower green line). However, for smokers (the red dots), there is a massive and sharp jump in costs across all ages. Age adds a slow increase, but smoking causes the sharp jump.

## 4. Cost Difference Between Genders Controlled by Smoking 

**Question:** Is there a cost difference between males and females, and does it disappear once we control for smoking status?

In [ ]:
sns.barplot(data=df, x='sex', y='charges', hue='smoker', palette={'yes': "#e0725fbb", 'no': "#3d7f99d0"})
plt.title('Average Charges by Gender and Smoking Status', fontsize=13, fontweight='bold')
plt.xlabel('Gender')
plt.ylabel('Average Charges ($)')
plt.tight_layout()
plt.show()

**Finding:** At a quick glance, males seem to cost more overall. However, once we split the data by smoking status, the difference between males and females almost entirely disappears. Males have higher total costs simply because a higher percentage of males are smokers, not because of their gender.